# 03 — Regressão Linear: Projeção de Receita

Este notebook aplica **regressão linear** (SciPy `linregress`) sobre o dataset financeiro para:
- Identificar a tendência de receita ao longo do período
- Projetar receita para os próximos **90 dias**
- Avaliar a qualidade do modelo via **R²** e erro padrão

O modelo é simples propositalmente — o objetivo é demonstrar o fluxo completo desde dados brutos até previsão documentada.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import linregress
import matplotlib.pyplot as plt
import os

os.makedirs("outputs", exist_ok=True)
print("Bibliotecas carregadas.")

## 3.1 — Preparar dados de receita

In [ ]:
df = pd.read_parquet("data/processed/finance_clean.parquet")

# Filtrar apenas receitas (excluir anomalias para treino limpo)
receita = df[(df["type"] == "receita") & (~df["anomalia"])].copy()

print(f"Registros de receita (sem anomalias): {len(receita)}")
receita.head()

In [ ]:
# Agregar receita por mês para suavizar
receita_mensal = (
    receita
    .set_index("date")
    .resample("ME")["amount"]
    .sum()
    .reset_index()
)
receita_mensal.columns = ["date", "receita_mensal"]

# Variável temporal: dias desde o início
receita_mensal["t"] = (receita_mensal["date"] - receita_mensal["date"].min()).dt.days

print(f"Meses com dados: {len(receita_mensal)}")
receita_mensal.head(10)

## 3.2 — Split treino/validação (80/20)

In [ ]:
split_idx = int(len(receita_mensal) * 0.8)

treino = receita_mensal.iloc[:split_idx]
valid  = receita_mensal.iloc[split_idx:]

print(f"Treino: {len(treino)} meses ({treino['date'].min().strftime('%Y-%m')} a {treino['date'].max().strftime('%Y-%m')})")
print(f"Validação: {len(valid)} meses ({valid['date'].min().strftime('%Y-%m')} a {valid['date'].max().strftime('%Y-%m')})")

## 3.3 — Regressão Linear (treino)

In [ ]:
slope, intercept, r_value, p_value, std_err = linregress(treino["t"], treino["receita_mensal"])

r_squared = r_value ** 2

print("=== Modelo de Regressão Linear ===")
print(f"Equação:      receita = {intercept:,.2f} + {slope:,.4f} × t")
print(f"R²  (treino): {r_squared:.4f}")
print(f"p-value:      {p_value:.6f}")
print(f"Erro padrão:  {std_err:,.4f}")
print(f"\nInterpretação: a cada dia, a receita mensal {'cresce' if slope > 0 else 'diminui'} em média R$ {abs(slope):,.2f}")

## 3.4 — Avaliação no conjunto de validação

In [ ]:
# Predição no conjunto de validação
valid = valid.copy()
valid["pred"] = intercept + slope * valid["t"]

# Métricas de validação
ss_res = ((valid["receita_mensal"] - valid["pred"]) ** 2).sum()
ss_tot = ((valid["receita_mensal"] - valid["receita_mensal"].mean()) ** 2).sum()
r2_valid = 1 - (ss_res / ss_tot) if ss_tot > 0 else float('nan')

mae = (valid["receita_mensal"] - valid["pred"]).abs().mean()
rmse = np.sqrt(((valid["receita_mensal"] - valid["pred"]) ** 2).mean())

print("=== Métricas de Validação ===")
print(f"R² (validação): {r2_valid:.4f}")
print(f"MAE:            R$ {mae:,.2f}")
print(f"RMSE:           R$ {rmse:,.2f}")

valid[["date", "receita_mensal", "pred"]]

## 3.5 — Projeção: próximos 90 dias

In [ ]:
# Gerar pontos futuros (3 meses = ~90 dias)
t_max = receita_mensal["t"].max()
t_futuro = np.array([t_max + 30, t_max + 60, t_max + 90])
datas_futuro = [receita_mensal["date"].max() + pd.DateOffset(months=i) for i in [1, 2, 3]]
projecao = intercept + slope * t_futuro

df_proj = pd.DataFrame({
    "date": datas_futuro,
    "t": t_futuro,
    "receita_projetada": projecao
})

print("=== Projeção (próximos 90 dias) ===")
df_proj

In [ ]:
# ── GRÁFICO PRINCIPAL ──
fig, ax = plt.subplots(figsize=(12, 5))

# Dados reais
ax.plot(receita_mensal["date"], receita_mensal["receita_mensal"],
        "o-", color="#3b82f6", markersize=5, label="Receita Real", linewidth=1.5)

# Linha de regressão sobre todo o período
ax.plot(receita_mensal["date"], intercept + slope * receita_mensal["t"],
        "--", color="#6b7280", linewidth=1, label="Regressão Linear (treino)", alpha=0.7)

# Split vertical
split_date = valid["date"].min()
ax.axvline(split_date, color="#f59e0b", linestyle=":", linewidth=1.2, label="Split treino/validação")

# Projeção futura
all_future_dates = [receita_mensal["date"].max()] + datas_futuro
all_future_vals = [intercept + slope * t_max] + list(projecao)
ax.plot(all_future_dates, all_future_vals,
        "s--", color="#f97316", markersize=7, linewidth=2, label="Projeção (90 dias)")

# Banda de incerteza na projeção (±1 RMSE)
ax.fill_between(datas_futuro,
                projecao - rmse, projecao + rmse,
                alpha=0.15, color="orange", label="Incerteza (±1 RMSE)")

# Formatação
ax.set_title("Projeção de Receita Mensal — Regressão Linear", fontsize=13, fontweight="bold")
ax.set_xlabel("Mês")
ax.set_ylabel("Receita (R$)")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.3)

# Anotação do R²
ax.annotate(f"R² treino = {r_squared:.3f}\nR² valid = {r2_valid:.3f}",
            xy=(0.98, 0.05), xycoords="axes fraction", ha="right",
            fontsize=9, bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

plt.tight_layout()
plt.savefig("outputs/projecao_receita.png", dpi=150, bbox_inches="tight")
plt.show()
print("✔ Gráfico salvo em outputs/projecao_receita.png")

## 3.6 — Análise com dados do Ibovespa (correlação)

In [ ]:
# Carregar Ibovespa processado e agregar por mês
df_ibov = pd.read_parquet("data/processed/ibovespa_clean.parquet")

ibov_mensal = (
    df_ibov
    .set_index("date")
    .resample("ME")["Close"]
    .mean()
    .reset_index()
)
ibov_mensal.columns = ["date", "ibov_media"]

# Merge com receita mensal
merged = receita_mensal.merge(ibov_mensal, on="date", how="inner")

if len(merged) >= 5:
    corr = merged["receita_mensal"].corr(merged["ibov_media"])
    print(f"Correlação Receita × Ibovespa: {corr:.3f}")
    
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(merged["ibov_media"], merged["receita_mensal"],
              color="#8b5cf6", s=40, alpha=0.7)
    ax.set_title(f"Correlação Receita × Ibovespa (r = {corr:.3f})", fontweight="bold")
    ax.set_xlabel("Ibovespa — Média Mensal (pontos)")
    ax.set_ylabel("Receita Mensal (R$)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("outputs/correlacao_ibov.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✔ Gráfico salvo em outputs/correlacao_ibov.png")
else:
    print("⚠ Dados insuficientes para análise de correlação.")

---
## Resumo — Regressão Linear

| Métrica | Valor |
|---------|-------|
| R² (treino) | Calculado acima |
| R² (validação) | Calculado acima |
| MAE | Calculado acima |
| Horizonte de projeção | 90 dias (3 meses) |

**Próximo:** `04_timeseries.ipynb` — Série temporal com média móvel e bandas de desvio-padrão.